### This notebook creates Risk Maps for microcuencas in Peru using the results of the impact chain calculations from the BD_ClimateRiesgo_IKI sql database

**Created:** 12/26/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 12/26/2025 by Sophia Bakar
 
**Status:** in progress

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\

**Objective:**   

**Compatibility:** 

**Packages:** numpy, pandas, geopandas, sqlite3, matplotlib 

**Further documentation:**  
 
**Inputs:** shapefile of subbasins, BD_RiesgoClimatico_IKI sql database

**Outputs:** 
 
**Assumptions:** 
 
**Future work:** 
 
**Notes:** This script queries the IKI Climate Risk SQL database to get the risk values for a particular impact chain, scenario, and user. Maps are generate using the Peru subbasins shapefile merged by COMID with the results from the impact chains.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import sqlite3
import os
import unicodedata
from matplotlib.colors import ListedColormap
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.lines import Line2D

In [2]:
# Connect to the SQLite database
# user = 'sgilson'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
conn = sqlite3.connect(db_path)

#output_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\RiskMaps_Results'    
output_path = r"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\RiskMaps_Results"

In [3]:
#subbasins_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp'
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index("COMID").to_crs("EPSG:32718")

In [4]:
subbasins_gdf['Cuenca'].unique()

array(['Cuenca Ilave', 'Cuenca Ramis', 'Cuenca Ica', 'Cuenca Mantaro',
       'Cuenca Santa', 'Cuenca Chicama', 'Cuenca Moche',
       'Cuenca Chancay-Lambayeque', 'Cuenca Olmos', 'Cuenca Cascajal',
       'Cuenca Piura', 'Cuenca Sama', 'Cuenca Nanay', 'Cuenca Zarumilla',
       'Cuenca Tumbes', 'Cuenca Chira', 'Cuenca Alto Marañon',
       'Cuenca Mayo', 'Cuenca Viru', 'Cuenca Huamansaña',
       'Cuenca Lacramarca', 'Cuenca Nepeña', 'Cuenca Urubamba',
       'Cuenca Chancay - Huaral', 'Cuenca Chillon', 'Cuenca Rimac',
       'Cuenca Lurin', 'Cuenca Pampas', 'Cuenca Camana',
       'Cuenca Quilca - Vitor - Chili', 'Cuenca Locumba',
       'Cuenca Caplina', 'Intercuenca 13155'], dtype=object)

In [4]:
cuencas_to_plot = ["Cuenca Tumbes", "Cuenca Zarumilla", "Cuenca Nanay", "Cuenca Ramis", "Cuenca Ilave"] # add more as needed

use_all_cuencas = False

In [5]:
# Parameters
# to use a subset of IC
#impact_chains = [1,2,6,7,8,9,10,11,12,13,14,15,16,17]
# Uncomment the following lines to automatically get all impact chains from the table in the future
query = "SELECT DISTINCT IcID FROM ImpactChain_Results"
impact_chains = [row[0] for row in conn.execute(query).fetchall()]

In [6]:
# Get impact chain metadata from database
impact_chains_df = pd.read_sql("SELECT IcID, Name, Sector FROM ImpactChains", conn)
# Create a dictionary: IcID -> (Name, Sector)
ic_metadata = dict(zip(impact_chains_df['IcID'], zip(impact_chains_df['Name'], impact_chains_df['Sector'])))


In [7]:
scenario_map = {
    3: {"title": "Línea Base", "tag": "LineaBase"},
    1: {"title": "Escenario Futuro CMIP6 85 2050", "tag": "FutureScenario"},
}
scenario_ids_to_plot = list(scenario_map.keys())


In [8]:
def clean_name(name):
    """
    Make folder-safe names (remove accents, spaces → _).
    """
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode()
    return name.replace(" ", "_")

In [9]:
# Risk classification settings
risk_bins = [-float('inf'), 0.2501, 0.50, 0.75, 1]  # include anything <0 in 'Bajo'
risk_labels = ['Bajo', 'Medio', 'Alto', 'Muy alto']
risk_colors = ['green', 'yellow', 'orange', 'red']  # Low → High

def classify_risk_fixed(series):
    """
    Classify a continuous risk series into fixed categories.
    Values below 0 are also 'Bajo'.
    """
    return pd.cut(
        series,
        bins=risk_bins,
        labels=risk_labels,
        include_lowest=True,
        right=True
    )


In [10]:
# Function to query risk values for a given impact chain and scenario
def get_results_df(impact_chain_id, wascn_id):
    query = """
    SELECT COMID, Peligro, Exposicion, Vulnerabilidad, Riesgo
    FROM ImpactChain_Results
    WHERE IcID = ? AND WaScnID = ?
    """
    df = pd.read_sql_query(query, conn, params=(impact_chain_id, wascn_id))
    df.set_index("COMID", inplace=True)
    return df

In [11]:
def add_north_arrow(ax, loc_x=0.92, loc_y=0.92, size=0.05):
    """
    Add a compact north arrow to a matplotlib axis.
    """
    ax.annotate(
        "",
        xy=(loc_x, loc_y),
        xytext=(loc_x, loc_y - size),
        arrowprops=dict(
            facecolor="black",
            edgecolor="black",
            width=3,
            headwidth=10,
            headlength=10
        ),
        xycoords=ax.transAxes
    )

    ax.text(
        loc_x,
        loc_y + 0.015,
        "N",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        transform=ax.transAxes
    )


In [12]:
def add_scale_bar(ax, location="lower left"):
    
    """
    Add a proper scalebar. Works when CRS units are meters (UTM).
    """
    scalebar = ScaleBar(
        dx=1,                 # 1 unit in data = 1 meter
        units="m",
        dimension="si-length",
        location=location,
        length_fraction=0.25, # visual size relative to axis
        pad = 0.15,
        box_alpha=0.6
    )
    ax.add_artist(scalebar)

In [13]:
def add_legend_row(ax_leg):
    """
    Draw a centered legend in a dedicated axis row (no overlap with maps).
    """
    ax_leg.axis("off")

    handles = [
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=color,
            markeredgecolor='black',
            markersize=14,   # <-- bigger dots
            label=label
        )
        for label, color in zip(risk_labels, risk_colors)
    ]

    leg = ax_leg.legend(
        handles=handles,
        labels=risk_labels,
        loc="center",
        ncol=4,
        frameon=False,
        handletextpad=0.8,
        columnspacing=2.0,
        fontsize=13,       # <-- bigger text
    )

    

In [14]:
# function to create a plot with 1 row and three columns for subplots of Peligro, Exposicion, and Vulnerabilidad
def plot_pev_1x3_classified(gdf, title, output_file):
    """
    1 row x 3 cols:
    Peligro, Exposición, Vulnerabilidad
    all classified using risk bins.
    Legend appears in its own row below (never overlaps).
    """

    gdf_plot = gdf.copy()

    # classify each component
    gdf_plot["Peligro_plot"] = classify_risk_fixed(gdf_plot["Peligro"])
    gdf_plot["Exposicion_plot"] = classify_risk_fixed(gdf_plot["Exposicion"])
    gdf_plot["Vulnerabilidad_plot"] = classify_risk_fixed(gdf_plot["Vulnerabilidad"])

    # enforce category order
    cat_type = pd.CategoricalDtype(categories=risk_labels, ordered=True)
    for c in ["Peligro_plot", "Exposicion_plot", "Vulnerabilidad_plot"]:
        gdf_plot[c] = gdf_plot[c].astype(cat_type)

    risk_cmap = ListedColormap(risk_colors)

    fig = plt.figure(figsize=(18, 7))
    gs = fig.add_gridspec(nrows=2, ncols=3, height_ratios=[20, 2])

    axes = [
        fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[0, 2]),
    ]

    ax_leg = fig.add_subplot(gs[1, :])  # legend row spans all columns

    panels = [
        ("Peligro", "Peligro_plot"),
        ("Exposición", "Exposicion_plot"),
        ("Vulnerabilidad", "Vulnerabilidad_plot"),
    ]

    for ax, (panel_title, col) in zip(axes, panels):

        gdf_plot.plot(
            column=col,
            ax=ax,
            cmap=risk_cmap,
            categorical=True,
            legend=False,
            missing_kwds={"color": "lightgrey", "label": "Sin datos"}
        )

        # More visible boundaries
        gdf_plot.boundary.plot(ax=ax, color="black", linewidth=1.0)

        ax.set_title(panel_title, fontsize=13)
        ax.set_axis_off()

        add_north_arrow(ax)
        add_scale_bar(ax, location="lower left")

    # Title
    fig.suptitle(title, fontsize=16, fontweight="bold")

    # Legend in dedicated row
    add_legend_row(ax_leg)


    fig.subplots_adjust(top=0.82, bottom=0.06, wspace=0.02, hspace=0.05)

    plt.savefig(output_file, dpi=300)
    plt.close()


In [15]:
def plot_risk_two_scenarios_classified(gdf_left, gdf_right, title, left_label, right_label, output_file):
    """
    1 row x 2 cols:
    Baseline (left) vs Future (right) overall risk map.
    Legend appears in its own row below (never overlaps).
    """

    def prep(gdf):
        g = gdf.copy()
        g["Risk_plot"] = classify_risk_fixed(g["Riesgo"])
        cat_type = pd.CategoricalDtype(categories=risk_labels, ordered=True)
        g["Risk_plot"] = g["Risk_plot"].astype(cat_type)
        return g

    left = prep(gdf_left)
    right = prep(gdf_right)

    risk_cmap = ListedColormap(risk_colors)

    fig = plt.figure(figsize=(16, 8))
    gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[20, 2])

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax_leg = fig.add_subplot(gs[1, :])

    for ax, gdf_plot, lab in zip([ax1, ax2], [left, right], [left_label, right_label]):

        gdf_plot.plot(
            column="Risk_plot",
            ax=ax,
            cmap=risk_cmap,
            categorical=True,
            legend=False,
            missing_kwds={"color": "lightgrey", "label": "Sin datos"}
        )

        # More visible boundaries
        gdf_plot.boundary.plot(ax=ax, color="black", linewidth=1.0)

        ax.set_title(lab, fontsize=13)
        ax.set_axis_off()

        add_north_arrow(ax)
        add_scale_bar(ax, location="lower left")

    fig.suptitle(title, fontsize=16, fontweight="bold")

    add_legend_row(ax_leg)

    fig.subplots_adjust(top=0.88, bottom=0.06, wspace=0.02)

    plt.savefig(output_file, dpi=300)
    plt.close()

In [16]:
# Get Spanish impact chain metadata from DB
impact_chains_df_es = pd.read_sql("SELECT IcID, Name_ES, Sector_ES FROM ImpactChains",conn)

# IcID -> (Spanish Name, Spanish Sector)
ic_metadata_es = dict(zip(impact_chains_df_es["IcID"],zip(impact_chains_df_es["Name_ES"], impact_chains_df_es["Sector_ES"])))

In [17]:
for ic in impact_chains:

    name, sector = ic_metadata_es.get(ic, ("Unknown", "Unknown"))

    for cuenca in cuencas_to_plot if not use_all_cuencas else subbasins_gdf["Cuenca"].unique():

        cuenca_clean = clean_name(cuenca)

        cuenca_output_path = os.path.join(output_path, f"{cuenca_clean}")
        os.makedirs(cuenca_output_path, exist_ok=True)

        cuenca_gdf = subbasins_gdf[subbasins_gdf["Cuenca"] == cuenca]

        if cuenca_gdf.empty:
            print(f"⚠️ Cuenca sin datos (geometría vacía): {cuenca}")
            continue

        # --- Build results for both scenarios so we can do 2-panel risk map ---
        scenario_results = {}

        for wascn_id in scenario_ids_to_plot:

            scenario_title = scenario_map[wascn_id]["title"]
            scenario_tag = scenario_map[wascn_id]["tag"]

            # --- Get results for this IcID + WaScnID ---
            results_df = get_results_df(ic, wascn_id)

            if results_df.empty:
                print(f"⚠️ Sin resultados en DB: IcID={ic}, WaScnID={wascn_id}")
                continue

            results_gdf = cuenca_gdf.join(results_df, how="left")

            if results_gdf.empty:
                print(f"⚠️ Resultado vacío después del join: IcID={ic}, WaScnID={wascn_id}, Cuenca={cuenca}")
                continue

            # skip if everything is NaN for all numeric panels
            cols_needed = ["Peligro", "Exposicion", "Vulnerabilidad", "Riesgo"]
            if results_gdf[cols_needed].isna().all().all():
                print(f"⚠️ All NaN: IcID={ic}, WaScnID={wascn_id}, Cuenca={cuenca}")
                continue

            scenario_results[wascn_id] = results_gdf

            # --- Title for figures ---
            title = (f"{name} – {sector}\n{scenario_title} – {cuenca}")

            # --- Output filenames ---
            output_file_pev = os.path.join(
                cuenca_output_path,
                f"IC{ic}_{scenario_tag}_PEV_{cuenca_clean}.png"
            )

            try:
                plot_pev_1x3_classified(gdf=results_gdf, title=title,output_file=output_file_pev)

            except Exception as e:
                print(f"❌ Error PEV plot: IcID={ic}, WaScnID={wascn_id}, Cuenca={cuenca}")
                print(f"   -> {type(e).__name__}: {e}")

            output_file_risk = os.path.join(cuenca_output_path,f"IC{ic}_{scenario_tag}_RIESGO_{cuenca_clean}.png")


        # --- Now create combined 2-panel overall risk plot (Baseline vs Future) ---
        if (1 in scenario_results) and (3 in scenario_results):

            combined_title = f"Riesgo ante {name} – {sector}\n{cuenca}"

            output_file_risk = os.path.join(cuenca_output_path,f"IC{ic}_RIESGO_{cuenca_clean}.png")

            try:
                plot_risk_two_scenarios_classified(
                    gdf_left=scenario_results[3],
                    gdf_right=scenario_results[1],
                    title=combined_title,
                    left_label=scenario_map[3]["title"],
                    right_label=scenario_map[1]["title"],
                    output_file=output_file_risk
                )
            except Exception as e:
                print(f"❌ Error RIESGO comparison plot: IcID={ic}, Cuenca={cuenca}")
                print(f"   -> {type(e).__name__}: {e}")